# Meeting 13: linear systems

**CS 301, Introduction to Data Science, NJIT.** Unit II, meeting 13, the first notebook.

A **linear system** is a set of equations that must hold together, each a sum of unknowns multiplied by numbers. This notebook introduces linear systems, solves them with NumPy, and shows their solutions geometrically: two lines that cross, three planes that meet at a point or along a line, two lines that are nearly parallel. It ends with a question about a walk on a graph that the meeting answers.

Do it **before meeting 13**, then the second notebook of the meeting, [*Solvers and sparse matrices*](https://colab.research.google.com/github/ikoutis/cs301-labs/blob/main/unit-2/notebooks/meeting-13-solvers.ipynb), and upload both.

## How this works

- **This notebook needs no GPU.** Leave its runtime as it is.
- **First** choose **Save a copy in Drive** in the **File** menu, to save your work, and run the cells in order, from the top. To run a cell, click it and press **Shift + Enter**.
- **You may work with your group** of meetings 11 and 12, two or three students. **Every student uploads their own copy.**
- **Four parts.** Each has code to run, a short exercise, marked <font color="#c00000"><b>Your turn</b></font>, and a **check** that tests it. If a check reports a problem, its message says what to look at. One picture is drawn in three dimensions: drag it to turn it.
- **Four cells ask for your own words**, marked <font color="#c00000"><b>Your thinking</b></font>: a few sentences that explain what you saw. **These cells, with four in the second notebook, are what your upload is graded on.** Write them yourself, even when your group worked together.
- **One post** on the discussion board of meeting 13, **before the meeting begins**, after Part 4. One member may post for the group, as long as **every post begins with the names of all the members of the group**. Under every image, write one sentence that says what the image shows. **A post earns the attention mark of this meeting for everyone it names.** A student who does not post earns it by attending the meeting.
- **Then do the second notebook of the meeting**, [*Solvers and sparse matrices*](https://colab.research.google.com/github/ikoutis/cs301-labs/blob/main/unit-2/notebooks/meeting-13-solvers.ipynb), and upload **both notebooks** to Canvas together, as the last section says.

<font color="#c00000" size="5"><b>Your turn.</b></font> Double-click this cell and replace this line with your name and the names of the other members of your group.

In [ ]:
# Setup. Run this cell first.
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import plotly.graph_objects as go

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)           # a source of random numbers
COLOURS = ["#2a78d6", "#eb6834", "#2a8a3a"]


def equation(row, rhs):
    """An equation with two unknowns, as text: 3x + 2y = 12."""
    a1, a2 = row
    sign = "+" if a2 >= 0 else "-"
    return f"{a1:g}x {sign} {abs(a2):g}y = {rhs:g}"


def draw_lines(ax, A, b, title, points=(), dashed=()):
    """Each row of A x = b with two unknowns is a line in the plane of the unknowns: draw them, and the solutions given."""
    x = np.linspace(-1, 5, 100)
    for k, (row, rhs) in enumerate(zip(A, b)):
        a1, a2 = row
        style = "--" if k in dashed else "-"
        if abs(a2) > 1e-12:
            ax.plot(x, (rhs - a1 * x) / a2, style, color=COLOURS[k % 3], label=equation(row, rhs))
        else:
            ax.axvline(rhs / a1, linestyle=style, color=COLOURS[k % 3], label=equation(row, rhs))
    for point in points:
        ax.plot(point[0], point[1], "ko", markersize=7)
        ax.annotate(f"({point[0]:.3g}, {point[1]:.3g})", point, xytext=(8, 6), textcoords="offset points")
    ax.set_xlim(-1, 5)
    ax.set_ylim(-1, 5)
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=9)
    ax.grid(True, color="#e1e0d9")


def draw_planes(A, b, point=None, line=None, title=""):
    """Each row of A x = b with three unknowns is a plane: draw the planes, and the solution point or line if given.
    The picture can be turned with the mouse."""
    s = np.linspace(-4, 6, 30)
    X, Y = np.meshgrid(s, s)
    fig = go.Figure()
    for k, (row, rhs) in enumerate(zip(A, b)):
        a1, a2, a3 = row
        Z = (rhs - a1 * X - a2 * Y) / a3           # the plane as a surface over the (x, y) grid; a3 is never 0 here
        fig.add_trace(go.Surface(x=X, y=Y, z=Z, showscale=False, opacity=0.55,
                                 colorscale=[[0, COLOURS[k]], [1, COLOURS[k]]],
                                 name=f"{a1:g}x + {a2:g}y + {a3:g}z = {rhs:g}", showlegend=True))
    if point is not None:
        fig.add_trace(go.Scatter3d(x=[point[0]], y=[point[1]], z=[point[2]], mode="markers",
                                   marker=dict(size=6, color="black"), name="the solution"))
    if line is not None:
        fig.add_trace(go.Scatter3d(x=line[:, 0], y=line[:, 1], z=line[:, 2], mode="lines",
                                   line=dict(color="black", width=7), name="a line of solutions"))
    fig.update_layout(title=title, width=760, height=560, margin=dict(l=0, r=0, t=40, b=0),
                      scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z", zaxis=dict(range=[-6, 10])))
    fig.show()


print("Setup is done.")

## <font color="#1967d2"><b>Part 1: Two unknowns, two equations</b></font>

An equation is **linear** when each unknown is multiplied by a number and the terms are added: `2x + y = 5` is linear; `x * y = 5` and `x ** 2 = 5` are not. One linear equation with two unknowns has many solutions: each x has a corresponding y = 5 - 2x. A **linear system** is several equations that must hold at the same time:

```
2x + y = 5
 x - y = 1
```

A solution is a pair (x, y) that satisfies both. **Elimination** finds it systematically: use one equation to remove an unknown from the other. Add the two equations: `3x = 6`, so `x = 2`; put x = 2 into the first: `y = 1`. Check: 2 · 2 + 1 = 5 and 2 - 1 = 1.

The same system can be written with a matrix. Put the numbers that multiply the unknowns in a matrix `A`, one row for each equation; the unknowns in a vector `x`; the right-hand sides in a vector `b`:

```
A = [[2,  1],       x = [x, y],       b = [5, 1]
     [1, -1]]
```

Then `A @ x` computes the left-hand sides of both equations at once, row by row, and the system becomes **`A @ x == b`**. To find x, use `np.linalg.solve(A, b)`. The next cell solves the system above.

In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, -1.0]])        # one row for each equation: the numbers that multiply x and y
b = np.array([5.0, 1.0])           # the right-hand sides
x = np.linalg.solve(A, b)
print("the solution x:", x)
print("A @ x:", A @ x, "  which is b again")

**Checking the solution.** `x = [2, 1]` is the pair x = 2, y = 1 that elimination found above. `A @ x` multiplies each row of A by the unknowns and adds, which is the left-hand side of each equation, and gives b back. `np.linalg.solve` uses elimination, which the second notebook applies to larger systems.

<font color="#c00000" size="5"><b>Your turn.</b></font> Write the system below as `A2` and `b2` in the next cell, in place of the `...`, and run the check.

```
3x + 2y = 12
 x - 4y = -10
```

In [ ]:
A2 = ...      # Your turn: a 2 by 2 array, one row for each equation, the numbers that multiply x and y
b2 = ...      # Your turn: the two right-hand sides

In [ ]:
for name, value in (("A2", A2), ("b2", b2)):
    assert value is not ..., f"Replace the ... after {name}, and run that cell again."
A2, b2 = np.asarray(A2, dtype=float), np.asarray(b2, dtype=float)
assert A2.shape == (2, 2), f"A2 has shape {A2.shape}; it must be 2 by 2: two equations, two unknowns."
assert b2.shape == (2,), f"b2 has shape {b2.shape}; it must hold two numbers, the right-hand sides."
expected = np.array([2.0, 3.0])
if np.allclose(A2.T @ expected, b2) and not np.allclose(A2 @ expected, b2):
    raise AssertionError("The rows and the columns of A2 are swapped: row 0 holds the numbers of the first equation, "
                         "3 and 2, and row 1 those of the second, 1 and -4.")
if not np.allclose(A2 @ expected, b2) and np.allclose(np.abs(A2[1]), [1, 4]):
    raise AssertionError("Check the signs of the second equation: x - 4y = -10 has -4 and -10.")
assert np.allclose(A2 @ expected, b2), (
    "A2 @ [2, 3] does not give b2. Row 0 of A2 holds 3 and 2, row 1 holds 1 and -4, and b2 holds 12 and -10.")
print("Correct: the solution is", np.linalg.solve(A2, b2), "and A2 @ x gives b2 back.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
draw_lines(axes[0], A, b, "the system of the example", points=[np.linalg.solve(A, b)])
draw_lines(axes[1], A2, b2, "your system", points=[np.linalg.solve(A2, b2)])
fig.tight_layout()
plt.show()

**The geometric interpretation.** An equation with two unknowns is a **line**: the set of all the points (x, y) that satisfy it. Two equations are two lines, and a solution satisfies both, so it is a point on both: where the lines cross. Two lines that are not parallel cross exactly once, and the system has exactly one solution. The solution set is the intersection of the two lines.

**One line for a system of any size.** Answer in the next cell, in two or three sentences: a system of 1,000 equations with 1,000 unknowns is also written `A @ x == b`. What are the rows of `A`, what is in `b`, and what does the product `A @ x` compute, row by row?

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

## <font color="#1967d2"><b>Part 2: Three unknowns, three planes</b></font>

With three unknowns, an equation such as `x + y + z = 6` is satisfied by a whole **plane** of points (x, y, z). A solution must lie on all three planes. In the next example they intersect at a single point. The cell solves the system and draws that point: drag the picture to turn it, and look at the point from several sides.

In [ ]:
A3 = np.array([[1.0, 1.0, 1.0],
               [1.0, -1.0, 2.0],
               [2.0, 1.0, -1.0]])
b3 = np.array([6.0, 5.0, 1.0])
x3 = np.linalg.solve(A3, b3)
print("the solution:", x3, "  A3 @ x3:", A3 @ x3)
draw_planes(A3, b3, point=x3, title="three planes, and the one point on all of them")

**The intersection.** Each plane is one equation; the solution, x = 1, y = 2, z = 3, is the one point that is on all three. The first two planes meet along a **line**. In this example the third intersects that line at one point, so the solution is unique.

**When the third equation adds nothing.** Replace the third equation by the sum of the first two: `2x + 0y + 3z = 11`. Every point on the line in which the first two planes meet satisfies it too, so the third plane contains the whole line, and the system has infinitely many solutions. The matrix is singular, so `np.linalg.solve` stops with the error *Singular matrix*.

<font color="#c00000" size="5"><b>Your turn.</b></font> In the next cell, replace the `...` with the third row of the matrix, the first row plus the second, entry by entry, and with the third right-hand side. Then run the check, which solves the system, reads the error, and draws the three planes with the line they share.

In [ ]:
third_row = [..., ..., ...]       # Your turn: the first row plus the second row, entry by entry
third_rhs = ...                   # Your turn: the first right-hand side plus the second

In [ ]:
assert ... not in third_row and third_rhs is not ..., "Replace every ... in the cell above, and run it again."
A_line = np.array([[1.0, 1.0, 1.0], [1.0, -1.0, 2.0], list(third_row)], dtype=float)
b_line = np.array([6.0, 5.0, float(third_rhs)])
assert np.allclose(A_line[2], A_line[0] + A_line[1]), (
    f"The third row is {A_line[2].tolist()}. It must be the first row plus the second, entry by entry: "
    f"{(A_line[0] + A_line[1]).tolist()}.")
assert np.isclose(b_line[2], b_line[0] + b_line[1]), (
    f"The third right-hand side is {b_line[2]:g}; it must be the sum of the first two, {b_line[0] + b_line[1]:g}. "
    "With another number the third plane is parallel to the line of the first two and misses it: no solution at all.")
try:
    np.linalg.solve(A_line, b_line)
    solved = True
except np.linalg.LinAlgError as error:
    solved = False
    print("NumPy refuses the system:", error)
assert not solved, "NumPy solved this system, which it should not: are the three rows the ones asked for?"
print(f"Correct: the three equations hold only {np.linalg.matrix_rank(A_line)} independent ones, "
      "and the planes share a line.")

direction = np.cross(A_line[0], A_line[1])                          # the line that the first two planes share
point = np.linalg.lstsq(A_line[:2], b_line[:2], rcond=None)[0]      # one point on it
t = np.linspace(-3, 3, 2)
line = point[None, :] + t[:, None] * direction[None, :] / np.linalg.norm(direction)
draw_planes(A_line, b_line, line=line, title="three planes that share a line: every point of the line is a solution")

**Rank and singular systems.** Only two of the three equations are independent: the third is a consequence of the other two. The number of equations that are independent of one another is the **rank** of the system, `np.linalg.matrix_rank(A)`. A system of n equations in n unknowns has exactly one solution when its rank is n, and then `solve` finds it. When the rank is smaller there are two possibilities. The planes share a line, as in the picture, and there are infinitely many solutions. Or, with another right-hand side (try 12 in place of 11), the third plane is parallel to that line and misses it, and there is no solution at all. NumPy says *Singular matrix* in both cases. With two unknowns the same two things happen with two lines: the same line twice, or two parallel lines.

**The pictures.** Answer in the next cell, in a few sentences: in the picture of two lines and in the picture of three planes, what does the solution of the system look like? What has happened in the picture when NumPy says *Singular matrix*? Describe both cases, infinitely many solutions and none.

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

## <font color="#1967d2"><b>Part 3: When the lines are nearly parallel</b></font>

A unique solution can be very sensitive to small changes in the data. The two systems in the next cell both have the solution x = 1, y = 1. In the first, the two lines cross at a right angle. In the second, they cross at a very small angle: `x + y = 2` and `x + 1.001y = 2.001` are nearly the same line. The cell changes the right-hand side of each system by a thousandth, as a measurement error would, and solves again.

In [ ]:
A_good = np.array([[1.0, 1.0], [1.0, -1.0]])
A_bad = np.array([[1.0, 1.0], [1.0, 1.001]])
x_true = np.array([1.0, 1.0])
change = np.array([0.0, 0.001])                      # a change of one thousandth in the second right-hand side

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
for ax, A_, name in ((axes[0], A_good, "lines that cross at a right angle"), (axes[1], A_bad, "lines that are nearly parallel")):
    b_ = A_ @ x_true
    x_moved = np.linalg.solve(A_, b_ + change)
    print(f"{name}:")
    print(f"   the solution {np.linalg.solve(A_, b_)}, after the change {x_moved}: moved by {np.linalg.norm(x_moved - x_true):.4f}")
    print(f"   condition number {np.linalg.cond(A_):,.0f}")
    draw_lines(ax, np.vstack([A_, A_[1]]), np.append(b_, (b_ + change)[1]), name, points=[x_true, x_moved], dashed=(2,))
fig.tight_layout()
plt.show()

**Sensitivity to changes in the data.** A change of 0.001 in the data moved the solution of the first system by 0.0007 and the solution of the second by 1.4: the same change, magnified about two thousand times. In the picture, the crossing point of two nearly parallel lines slides far along them when one line moves a little (the dashed line is the moved one; it lies on top of the line it came from). The number that measures this is the **condition number** of the system, `np.linalg.cond(A)`: an upper bound on the amplification of relative changes in b. The smallest possible condition number is 1; a condition number of 10^k can cost up to k of the digits of the solution. Errors can come from measured data and from floating-point rounding. With six-digit data and condition number 10^6, no reliable solution digits are guaranteed. The actual loss depends on the error in the data.

<font color="#c00000" size="5"><b>Your turn.</b></font> In the next cell, change the second row of the matrix so that the condition number is at least a million, without making the two lines the same line. Then run the check.

In [ ]:
second_row = [1.0, ...]      # Your turn: replace the ... so that the two lines are nearly, but not exactly, the same

In [ ]:
assert ... not in second_row, "Replace the ... in the cell above, and run it again."
A_ill = np.array([[1.0, 1.0], list(second_row)], dtype=float)
condition = np.linalg.cond(A_ill)
if not np.isfinite(condition) or condition > 1e15:
    raise AssertionError("That makes the second line the same line as the first, or parallel to it: the system is singular, "
                         "not nearly so. Choose a number close to 1, such as 1.000001, but not 1.")
assert condition >= 1e6, (f"The condition number is {condition:,.0f}; it must be at least 1,000,000. "
                          "The closer the second row is to the first, the larger it is.")
b_ill = A_ill @ np.array([1.0, 1.0])
x_moved = np.linalg.solve(A_ill, b_ill + np.array([0.0, 0.001]))
print(f"Correct: the condition number is {condition:,.0f}. A change of 0.001 in b moves the solution from [1, 1] to {x_moved}.")

**Measured data.** Answer in the next cell, in a few sentences: the right-hand sides of a system often come from measurements, good to a few digits. Why is a system whose lines are nearly parallel sensitive to those errors, and what does the condition number tell you about the possible solution error?

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

## <font color="#1967d2"><b>Part 4: A walk on a graph</b></font>

A **graph** is a set of **nodes** joined by **edges**: the pages of a website joined by their links, for instance. The next cell draws a graph of 16 nodes with two of them marked, one **red** and one **blue**. A particle starts at a node and, at every step, moves along an edge to a neighbour chosen at random, until it reaches the red node or the blue node. From each starting node, which of the two does it reach first, and how often?

The cell after the drawing lets the particle walk 2,000 times from every node and colours each node by the fraction of the walks that reached red first: 1 means always red, 0 always blue.

In [ ]:
G = nx.random_geometric_graph(16, 0.4, seed=7)       # 16 nodes placed at random; an edge joins two nodes closer than 0.4
positions = nx.get_node_attributes(G, "pos")
lengths = dict(nx.all_pairs_shortest_path_length(G))
RED, BLUE = max(((u, v) for u in G for v in G if u < v), key=lambda pair: lengths[pair[0]][pair[1]])  # the two farthest apart
neighbours = {v: list(G.neighbors(v)) for v in G}

print(f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges. Red: node {RED}. Blue: node {BLUE}.")
print("the neighbours of each node:", {v: neighbours[v] for v in sorted(G)})
fig, ax = plt.subplots(figsize=(7, 6))
node_colours = ["#d62728" if v == RED else "#1f77b4" if v == BLUE else "#dddddd" for v in G]
nx.draw(G, positions, ax=ax, node_color=node_colours, with_labels=True, node_size=520, font_size=10, edge_color="#999999")
ax.set_title("a graph of 16 nodes; the walk stops at the red node or at the blue node")
plt.show()

In [ ]:
def red_first(start, walks=2000):
    """The fraction of `walks` random walks from `start` that reach the red node before the blue node."""
    reached_red = 0
    for _ in range(walks):
        v = start
        while v != RED and v != BLUE:
            v = neighbours[v][rng.integers(len(neighbours[v]))]      # a neighbour chosen at random
        reached_red += v == RED
    return reached_red / walks


fractions = {v: red_first(v) for v in sorted(G)}
print("the fraction of the walks that reached red first, from each node:")
print(", ".join(f"{v}: {fractions[v]:.2f}" for v in fractions))

fig, ax = plt.subplots(figsize=(7, 6))
nx.draw(G, positions, ax=ax, node_color=[fractions[v] for v in G], cmap="coolwarm", vmin=0, vmax=1,
        with_labels=True, node_size=520, font_size=10, edge_color="#999999")
for v in G:
    ax.annotate(f"{fractions[v]:.2f}", positions[v], xytext=(0, -18), textcoords="offset points", ha="center", fontsize=8)
ax.set_title("the fraction of the walks that reached red first: blue is 0, red is 1")
plt.show()

print("Line for the discussion board:")
print("Red first, from each node: " + ", ".join(f"{v}: {fractions[v]:.2f}" for v in fractions)
      + f". Red is node {RED}, blue is node {BLUE}.")

**Estimated probabilities.** Every node has a number between 0 and 1: near the red node the walks mostly end red, near the blue node mostly blue, and in between the numbers change step by step along the edges. With 2,000 walks, sampling errors are typically on the order of a few hundredths. More trials generally improve the estimates.

**Bring this question to the meeting.** The sixteen probabilities also satisfy a linear system, with one unknown per node. Solving it avoids simulation error. What are its equations? We will discuss your proposals in class and relate the walk to linked web pages.

**Your guess.** Answer in the next cell, in a few sentences: find the node whose fraction is nearest to one half and the node whose fraction is nearest to 1, and say why the walks from them end as they do. Then guess: how could the fraction of a node be computed from the fractions of its neighbours?

<font color="#c00000" size="5"><b>Your thinking.</b></font> Double-click this cell and replace this line with your answer, in your own words.

<font color="#c00000" size="5"><b>Post now</b></font>. One member posts for the group on the discussion board of meeting 13. Begin the post with the names of all the members of your group: the post earns the attention mark of this meeting for everyone it names. The post contains:

- the picture of the three planes of Part 2 with the solution point (the camera icon at the top right of the picture saves it as an image), and one sentence that says what it shows;
- the picture of Part 3, the lines that cross at a right angle and the lines that are nearly parallel, with the two condition numbers printed above it, and one sentence;
- the coloured graph of Part 4, the line for the discussion board printed under it, and one sentence: your guess from the last cell.

## Then: the second notebook, and the upload

**Next, do the second notebook of meeting 13, [*Solvers and sparse matrices*](https://colab.research.google.com/github/ikoutis/cs301-labs/blob/main/unit-2/notebooks/meeting-13-solvers.ipynb).** It solves systems of thousands of equations three ways and times them, and introduces sparse storage for matrices with many zero entries.

**Every student uploads their own copies of both notebooks, before the meeting begins.** They are graded on the cells marked <font color="#c00000"><b>Your thinking</b></font> and <font color="#c00000"><b>Your prompt</b></font>: four in this notebook and four in the second. Replace the placeholder text in each graded cell with your own response.

1. In each notebook, open the **File** menu, choose **Download**, and then **Download .ipynb**.
2. On Canvas, open the module of meeting 13, choose **Upload Your Class Work - Meeting 13**, and upload **both files** in the same submission.